# FaceProof — resolution ablation (Threats to Validity)

**Purpose.** Our headline finding is that every detector collapses *below chance* on modern
text-to-image faces (SDXL / Flux / DALL-E 3). One alternative explanation is a **preprocessing
artifact**: the real faces originate at ~256px (FFHQ / "140k") while the T2I fakes originate at
~1024px, so the two classes reach the 224px crop through *different* amounts of downscaling. A
detector could latch onto that resolution asymmetry instead of synthesis cues.

**This notebook controls for it.** We re-preprocess the T2I fakes through a **256px intermediate**
(matching the reals' native resolution) *before* the standard 224 + JPEG-90 pipeline, then re-score
the **saved** C4 (CLIP) and C1 (ResNet) probes against the same held-out reals. We print the
original vs resolution-matched AUROC side by side.

**Read the result:**
- matched AUROC still `<< 0.50` -> the below-chance collapse is **not** a resolution artifact (it
  survives the control); the finding stands.
- matched AUROC rises toward/above `0.50` -> resolution asymmetry was (partly) responsible; a real
  threat to the finding.

Either way we report it honestly in `reports/report.md` (Threats to Validity). This notebook does
**not** touch `results.csv` (frozen numbers) — it writes a separate `reports/ablation_resolution.md`.

> Self-contained, **GPU on**. Reuses `src.preprocessing`, `src.features`, and the saved probes.

## 1. Setup + data (GPU on)

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q kaggle open_clip_torch scikit-learn joblib pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT         = Path('/content/drive/MyDrive/faceproof')
PROBES_ROOT  = ROOT / 'models' / 'probes'
REPORTS_ROOT = ROOT / 'reports'
MANIFEST     = ROOT / 'data' / 'manifest.csv'
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import glob
from google.colab import files
# Skip the (large) download if the SFHQ-T2I CSV is already present.
def find_t2i_csv():
    cs = [c for c in glob.glob('/content/**/*.csv', recursive=True) if 't2i' in c.lower() or 'sfhq' in c.lower()]
    return cs[0] if cs else None
if find_t2i_csv() is None:
    print('Upload kaggle.json:'); files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d selfishgene/sfhq-t2i-synthetic-faces-from-text-2-image-models -p /content/t2i --unzip
CSV = find_t2i_csv(); assert CSV, 'T2I CSV not found'
print('T2I CSV:', CSV)

In [ ]:
meta = pd.read_csv(CSV)
COL_IMAGE, COL_GEN = 'image_filename', 'model_used'   # confirmed SFHQ-T2I schema
img_root = Path(CSV).parent
index = {q.name: str(q) for q in img_root.rglob('*') if q.suffix.lower() in {'.jpg','.jpeg','.png'}}
meta['path'] = meta[COL_IMAGE].map(index.get)
meta = meta.dropna(subset=['path'])
meta = meta[meta[COL_GEN].isin(['SDXL','FLUX1_schnell','DALLE3'])].copy()   # 3 distinct families
assert len(meta), 'meta empty - check columns/download'
print(meta[COL_GEN].value_counts())

## 2. Load probes, held-out reals, and real features (computed once)

The reals are **identical** in both conditions (they already originate at ~256px), so only the
*fake* preprocessing changes between original and matched. We therefore extract the real features
once and reuse them.

In [ ]:
import yaml, joblib, shutil
from PIL import Image
from src.preprocessing import preprocess_image
from src.features import extract_clip, extract_resnet
from src.probe import predict_proba
from src.metrics import auroc

dcfg = yaml.safe_load(open('configs/data.yaml')); IMG, Q = dcfg['image_size'], dcfg['jpeg_match_quality']
mcfg = yaml.safe_load(open('configs/model.yaml'))['clip']
INTERMEDIATE = 256   # FFHQ / "140k" real origin; fakes are forced through this before 224/JPEG-90

probes = {c: joblib.load(PROBES_ROOT / f'{c}.joblib') for c in ['c1_resnet','c4_clip']}

_mani = pd.read_csv(MANIFEST)
real_paths = _mani[(_mani.label==0) & (_mani['split']=='test_indist')]['path'].tolist()[:3000]
assert real_paths, 'no held-out reals in manifest'

# Reals are the same in both conditions -> extract their features once.
Xr_clip = extract_clip(real_paths, backbone=mcfg['backbone'], pretrained=mcfg['pretrained'], batch_size=64, cache=None)
Xr_res  = extract_resnet(real_paths, batch_size=64, cache=None)
print('real features:', Xr_clip.shape, Xr_res.shape)

## 3. Preprocess fakes two ways + score the saved probes

`pre_size=None` = the original pipeline (fake 1024 -> 224). `pre_size=256` forces a 256px
intermediate first (fake 1024 -> 256 -> 224), so the fakes now traverse the *same* 256 -> 224
downscale the reals do.

In [ ]:
def crops_for(paths, tag, pre_size=None):
    """Compression-match fakes to 224/JPEG-Q. If pre_size is set, force that intermediate
    resolution first (matches the reals' origin), isolating resolution asymmetry."""
    out = Path(f'/content/_abl_{tag}')
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    kept = []
    for i, p in enumerate(paths):
        src = p
        if pre_size is not None:
            im0 = Image.open(p).convert('RGB').resize((pre_size, pre_size), Image.BICUBIC)
            src = out / f'_src_{i:06d}.png'; im0.save(src, 'PNG')   # lossless intermediate
        im = preprocess_image(src, IMG, Q, detector=None)
        if im is None: continue
        op = out / f'{i:06d}.jpg'; im.save(op, 'JPEG', quality=Q); kept.append(str(op))
    return kept

def eval_fakes(fake_paths):
    Xf_clip = extract_clip(fake_paths, backbone=mcfg['backbone'], pretrained=mcfg['pretrained'], batch_size=64, cache=None)
    Xf_res  = extract_resnet(fake_paths, batch_size=64, cache=None)
    res = {}
    for cond, (Xr, Xf) in {'c4_clip': (Xr_clip, Xf_clip), 'c1_resnet': (Xr_res, Xf_res)}.items():
        Xall = np.vstack([Xr, Xf]); yall = np.r_[np.zeros(len(Xr)), np.ones(len(Xf))]
        res[cond] = round(auroc(yall, predict_proba(probes[cond], Xall)), 4)
    return res

rows = []
for gen, grp in meta.groupby(COL_GEN):
    paths = grp['path'].tolist()[:3000]
    orig    = eval_fakes(crops_for(paths, gen + '_orig'))
    matched = eval_fakes(crops_for(paths, gen + '_m256', pre_size=INTERMEDIATE))
    rows.append({'generator': gen, 'n_fake': len(paths),
                 'C4_orig': orig['c4_clip'], 'C4_matched': matched['c4_clip'],
                 'C1_orig': orig['c1_resnet'], 'C1_matched': matched['c1_resnet']})
    print(f"{gen}: C4 {orig['c4_clip']}->{matched['c4_clip']} | C1 {orig['c1_resnet']}->{matched['c1_resnet']}")

abl = pd.DataFrame(rows)
print('\n' + abl.to_string(index=False))

In [ ]:
# Save a small markdown table (separate from the frozen results.csv) to fold into the report.
cols = list(abl.columns)
hdr  = '| ' + ' | '.join(cols) + ' |'
sep  = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
body = ['| ' + ' | '.join(str(v) for v in r) + ' |' for r in abl.values]
table_md = '\n'.join([hdr, sep] + body)

out_md = REPORTS_ROOT / 'ablation_resolution.md'
out_md.write_text(
    '# Resolution ablation - T2I AUROC: original vs 256px-matched preprocessing\n\n'
    'Saved C4 (CLIP) / C1 (ResNet) probes vs held-out reals. Chance = 0.50.\n\n'
    + table_md +
    '\n\n**Reading:** matched AUROC still << 0.50 => below-chance collapse is NOT a resolution '
    'artifact (survives the control). Matched AUROC rising to ~0.50+ => resolution asymmetry '
    'contributed to the collapse.\n'
)
print('saved', out_md)
print('\nPaste the table above back so it can be folded into reports/report.md (Threats to Validity).')